# noise_reachability step 3b: is there a noise signal a critic can learn? (Kaggle, 2 x T4)

Step 3b of `noise_reachability/README.md`. The base policy (`lerobot/smolvla_libero` at `a24f1f3804`,
`n_action_steps=10`) runs task 7 on init states 10-49 and logs, per inference, its features and the N(0, 1)
noise it executed: 4 streams (`--seed` 0-3) x 250 episodes, two streams per T4, through
`noise_reachability/run_step3_noise_sac.sh` with every episode a warm-up episode and no updates. Step 11 then
fits Monte Carlo critics and reports the held-out action AUC (`analyze_noise_signal.py`).

**Run it with Save & Run All (Accelerator: GPU T4 x2, Internet on).** Nothing needs editing.

**A session ends after 12 hours.** Step 10 stops the streams at `STOP_AFTER_H` so the version saves its
output; a stream loses at most the 25 episodes since its last save. If step 10 prints that episodes remain,
add this version's output as input (**Add Input > Your Work > this notebook's output**) and Save & Run All
again: step 6 copies it in and the streams resume. Every version must clone the same commit (step 6 checks).

Downloads (Output tab): `step3b_results.zip` (logs, report; small) and, when all episodes are done,
`step3b_replay.zip` (the replay parts, for re-analysis).

## 1. Settings

In [ ]:
import csv, glob, json, os, re, shutil, signal, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/swagatk/efficient_vla.git"
REPO_REF = "main"
REPO = "/tmp/efficient_vla"        # code and venv live outside /kaggle/working, so they are not saved as output
VENV = "/tmp/venv"
PY = f"{VENV}/bin/python"

# The protocol of README.md step 3b; nothing here is meant to be edited.
CHECKPOINT_REPO, CHECKPOINT_REVISION = "lerobot/smolvla_libero", "a24f1f380461b665158d52acd140152c79e8b36d"
TASK = 7
STREAM_SEEDS = [0, 1, 2, 3]
EPISODES = 250                     # per stream
NOISE_SCALE = 5.0                  # the clip at 0.999 x 5 changes no entry in practice: the base's own noise
COLLECT_ARGS = (f"--num_episodes {EPISODES} --warmup_episodes {EPISODES} --updates_per_decision 0 "
                f"--noise_scale {NOISE_SCALE}")
OUT_NAME = "noise_signal_step3b"
OUT_ROOT = f"/kaggle/working/{OUT_NAME}"
STOP_AFTER_H = 9.5                 # stop the streams this long after step 10 starts; Kaggle ends a session at 12 h,
                                   # and the analysis (step 11) needs about an hour (estimate)
GB_PER_JOB = 5                     # one SmolVLA process held about 4.5 GB of RAM locally

# Commits of the editable installs in the local lerobot_env that produced the existing results.
LEROBOT_COMMIT = "b6ec0060779550c0a157ae34feb89e0cf86012a8"
LIBERO_COMMIT = "8f1084e3132a39270c3a13ebe37270a43ece2a01"
print(f"step 3b: task {TASK}, streams (seeds) {STREAM_SEEDS} x {EPISODES} episodes, {COLLECT_ARGS}, "
      f"output {OUT_ROOT}")


def sh(cmd, check=True, env=None):
    # Capture and print: notebooks do not show output that a subprocess writes directly.
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr).rstrip(), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r

## 2. Hardware

In [ ]:
smi = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader")
gpus = [line.split(", ") for line in smi.stdout.strip().splitlines()]
assert gpus, "No GPU. Set Accelerator to GPU T4 x2."
for name, mem, cap, driver in gpus:
    assert float(cap) >= 7.5, f"{name} (compute capability {cap}) is not supported by the pinned PyTorch build; use GPU T4 x2"
CUDA_DRIVER = float(re.search(r"CUDA Version: ([0-9.]+)", sh("nvidia-smi").stdout).group(1))
mem_gb = int(open("/proc/meminfo").read().split()[1]) / 1e6
avail_gb = int(re.search(r"MemAvailable:\s+(\d+)", open("/proc/meminfo").read()).group(1)) / 1e6
JOBS = max(1, min(2 * len(gpus), int((avail_gb - 4) // GB_PER_JOB), os.cpu_count()))
GPU_IDS = ",".join(str(i) for i in range(len(gpus)))
print(f"{len(gpus)} GPU(s), driver supports CUDA {CUDA_DRIVER}; {os.cpu_count()} CPU cores; "
      f"RAM {mem_gb:.0f} GB ({avail_gb:.0f} GB available) -> {JOBS} evaluations at a time on GPUs {GPU_IDS}")
assert len(gpus) == 2, "Set Accelerator to GPU T4 x2"
assert JOBS >= len(STREAM_SEEDS), f"RAM for {JOBS} SmolVLA processes, the protocol runs {len(STREAM_SEEDS)} at once"

## 3. Code

In [ ]:
if not Path(REPO, ".git").exists():
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch origin && git -C {REPO} checkout {REPO_REF} && git -C {REPO} pull --ff-only origin {REPO_REF}")
sh(f"git -C {REPO} log -1 --format='%H %s'")

## 4. Environment

The package versions of the local `lerobot_env` (as the step-0/1 Kaggle notebook). Takes several minutes.

In [ ]:
# Headless rendering: MuJoCo needs the NVIDIA EGL vendor file, which some images lack.
icd = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}))
sh("apt-get install -y -qq libegl1 libgles2 libgl1 > /dev/null", check=False)

sh("python -m pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv {VENV} --python 3.12")

req = Path(REPO, "hybrid_diffusion", "requirements-colab.txt").read_text().splitlines()
if CUDA_DRIVER >= 13.0:
    TORCH_BUILD = "cu130 (as pinned)"
    reqs = req
else:
    cu = "cu128" if CUDA_DRIVER >= 12.8 else "cu126"
    assert CUDA_DRIVER >= 12.6, f"driver supports only CUDA {CUDA_DRIVER}; need 12.6 or newer"
    TORCH_BUILD = f"{cu} (driver too old for cu130)"
    torch_pins = [l for l in req if l.split("==")[0] in ("torch", "torchvision")]
    sh(f"uv pip install -q --python {PY} " + " ".join(f"'{p}+{cu}'" for p in torch_pins)
       + f" --index-url https://download.pytorch.org/whl/{cu} --extra-index-url https://pypi.org/simple")
    reqs = [l for l in req if not re.match(r"(torch|torchvision|triton|nvidia-)", l)]
print("PyTorch build:", TORCH_BUILD)
Path("/tmp/requirements.txt").write_text("\n".join(reqs) + "\n")
sh(f"uv pip install -q --python {PY} --no-deps -r /tmp/requirements.txt")
sh(f"uv pip install -q --python {PY} --no-deps 'lerobot @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}'")

LIBERO = os.path.expanduser("~/LIBERO")
if not Path(LIBERO, ".git").exists():
    sh(f"git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git {LIBERO}")
sh(f"git -C {LIBERO} checkout -q {LIBERO_COMMIT}")
Path(LIBERO, "libero", "__init__.py").touch()   # present in the local checkout, missing upstream
sh(f"uv pip install -q --python {PY} --no-deps -e {LIBERO}")

# LIBERO asks for these paths interactively on first import unless the config already exists.
root = f"{LIBERO}/libero/libero"
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
os.makedirs(f"{LIBERO}/libero/datasets", exist_ok=True)
Path(os.path.expanduser("~/.libero/config.yaml")).write_text(
    f"benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n"
    f"datasets: {LIBERO}/libero/datasets\nassets: {root}/assets\n"
)

## 5. Sanity checks and the checkpoint

PyTorch on both GPUs with bfloat16, headless MuJoCo (EGL), the CPU tests, and the pinned checkpoint
downloaded once.

In [ ]:
RUN_ENV = dict(os.environ, PATH=f"{VENV}/bin:" + os.environ["PATH"], MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl",
               MPLBACKEND="Agg", WANDB_MODE="disabled", OMP_NUM_THREADS="1", PYTHONPATH=REPO, PYTHON_BIN=PY)
sh(f"""{PY} -c "
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
x = torch.randn(256, 256, device='cuda', dtype=torch.bfloat16)
print('bfloat16 matmul ok', float((x @ x).float().abs().mean()) > 0)
" """, env=RUN_ENV)
egl = sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('EGL context ok')\"", check=False, env=RUN_ENV)
assert "EGL context ok" in egl.stdout, "EGL rendering failed"
sh(f"cd {REPO} && {PY} -m pytest tests/ -q -p no:cacheprovider", env=RUN_ENV)
r = sh(f"""{PY} -c "from huggingface_hub import snapshot_download; print(snapshot_download('{CHECKPOINT_REPO}', revision='{CHECKPOINT_REVISION}'))" """, env=RUN_ENV)
POLICY_PATH = r.stdout.strip().splitlines()[-1]
print("checkpoint", POLICY_PATH)

## 6. Resume from an earlier version's output

Copies an earlier version's streams in (checkpoints, replay parts, logs), and refuses to resume on code other
than the commit the first version cloned. Nothing happens on a first run.

In [ ]:
os.makedirs(OUT_ROOT, exist_ok=True)
for prev in sorted(glob.glob(f"/kaggle/input/**/{OUT_NAME}", recursive=True)):
    print("copying", prev)
    shutil.copytree(prev, OUT_ROOT, dirs_exist_ok=True)
head = sh(f"git -C {REPO} log -1 --format='%H'").stdout.strip()
first = Path(OUT_ROOT, "repo_commit.txt")
started_with = first.read_text().split()[0] if first.exists() and first.read_text().strip() else None
if started_with and started_with != head:
    raise RuntimeError(f"the run started at commit {started_with}, this version cloned {head}: set REPO_REF in "
                       f"step 1 to {started_with}")


def episodes_done(seed):
    f = Path(OUT_ROOT, f"seed_{seed}", f"task_{TASK}", "train_log.csv")
    ck = Path(OUT_ROOT, f"seed_{seed}", f"task_{TASK}", "checkpoint.pt")
    rows = list(csv.DictReader(open(f))) if f.exists() else []
    return len(rows), ck.exists()


DONE = {s: episodes_done(s)[0] for s in STREAM_SEEDS}
print("episodes logged per stream:", DONE, f"of {EPISODES} (the saved checkpoint decides where a stream resumes)")

## 7. Real-model check on this machine

On one observation of task 7: no chosen noise reproduces the base chunk bit for bit, the drawn noise passed
back unchanged does too, a fixed noise repeats exactly, and the features have the expected size
(`check_step3_steering.py`, as step 3).

In [ ]:
r = sh(f"cd {REPO} && {PY} -u noise_reachability/check_step3_steering.py --task_id {TASK} --policy_path {POLICY_PATH}",
       check=False, env=RUN_ENV)
assert r.returncode == 0 and "[check] all passed" in r.stdout, "a steering check failed; see the output above"

## 8. Smoke run (2 episodes, not counted)

Two collection episodes through the runner in a separate folder, in two processes of one episode each (so
the resume path runs too): the log, the replay part of each process, no updates, and the settings in
`provenance.jsonl`. Skipped when resuming.

In [ ]:
if sum(DONE.values()) == 0:
    smoke = "/tmp/step3b_smoke"
    shutil.rmtree(smoke, ignore_errors=True)
    t0 = time.time()
    r = sh(f"bash {REPO}/noise_reachability/run_step3_noise_sac.sh", check=False,
           env=dict(RUN_ENV, OUT_ROOT=smoke, TASKS=str(TASK), STAGES="train",
                    EXTRA_ARGS=f"--seed 9 --policy_path {POLICY_PATH} --num_episodes 2 --warmup_episodes 2 "
                               f"--updates_per_decision 0 --noise_scale {NOISE_SCALE} --episodes_per_process 1"))
    d = Path(smoke, f"task_{TASK}")
    log = list(csv.DictReader(open(d / "train_log.csv"))) if (d / "train_log.csv").exists() else []
    prov = [json.loads(l) for l in open(d / "provenance.jsonl")] if (d / "provenance.jsonl").exists() else []
    parts = sorted(p.name for p in d.glob("replay_part_*.npz"))
    ok = (r.returncode == 0 and [int(x["episode"]) for x in log] == [0, 1]
          and all(x["warmup"] == "1" and x["updates"] == "0" for x in log)
          and parts == ["replay_part_00000-00000.npz", "replay_part_00001-00001.npz"] and len(prov) == 2
          and all(p["settings"]["noise_scale"] == NOISE_SCALE and p["settings"]["updates_per_decision"] == 0
                  and p["settings"]["policy_path"] == POLICY_PATH and p["settings"]["n_action_steps"] == 10
                  and not p["git_dirty"] for p in prov))
    assert ok, "smoke run did not finish as expected; see the output above"
    print("smoke run ok: episodes (success, steps, decisions, s):",
          [(x["success"], x["steps"], x["decisions"], x["time_s"]) for x in log],
          f"| {(time.time() - t0) / 60:.1f} min including two model loads")
else:
    print("resuming; smoke run skipped")

## 9. Launch the streams

One runner per stream (`run_step3_noise_sac.sh`, `--seed` 0-3), streams 0 and 2 on GPU 0, 1 and 3 on GPU 1,
started a minute apart (EGL start-up, FAIL-006). Each runner works in processes of 25 episodes and resumes
from its last save.

In [ ]:
sh(f"git -C {REPO} log -1 --format='%H' >> '{OUT_ROOT}/repo_commit.txt'")
procs = {}
for k, seed in enumerate(STREAM_SEEDS):
    out = f"{OUT_ROOT}/seed_{seed}"
    os.makedirs(out, exist_ok=True)
    env = dict(RUN_ENV, OUT_ROOT=out, TASKS=str(TASK), STAGES="train", CUDA_VISIBLE_DEVICES=str(k % 2),
               EXTRA_ARGS=f"--seed {seed} --policy_path {POLICY_PATH} {COLLECT_ARGS}")
    log = open(f"{out}/runner.log", "a")
    procs[seed] = subprocess.Popen(f"bash {REPO}/noise_reachability/run_step3_noise_sac.sh", shell=True,
                                   executable="/bin/bash", env=env, stdout=log, stderr=subprocess.STDOUT,
                                   start_new_session=True)
    print(f"stream {seed}: GPU {k % 2}, pid {procs[seed].pid}, log {out}/runner.log", flush=True)
    if k < len(STREAM_SEEDS) - 1:
        time.sleep(60)

## 10. Progress (every 10 minutes) and the stop before the session limit

Runs in the foreground so Save & Run All waits for it. At `STOP_AFTER_H` the streams are stopped; their last
saves are kept and a new version resumes from them.

In [ ]:
T_START = time.time()


def progress():
    total, secs = 0, []
    for seed in STREAM_SEEDS:
        f = Path(OUT_ROOT, f"seed_{seed}", f"task_{TASK}", "train_log.csv")
        rows = list(csv.DictReader(open(f))) if f.exists() else []
        total += len(rows)
        secs += [float(r["time_s"]) for r in rows[-20:]]
        print(f"  stream {seed}: {len(rows)}/{EPISODES} episodes, {sum(int(r['success']) for r in rows)} ok", flush=True)
    left = len(STREAM_SEEDS) * EPISODES - total
    if secs:
        per = sum(secs) / len(secs)
        print(f"  {per:.0f} s per episode per stream -> about {left * per / len(STREAM_SEEDS) / 3600:.1f} h left", flush=True)
    return left


while any(p.poll() is None for p in procs.values()):
    time.sleep(600)
    print(time.strftime("%H:%M"), f"{(time.time() - T_START) / 3600:.1f} h", flush=True)
    progress()
    if time.time() - T_START > STOP_AFTER_H * 3600:
        print("stopping the streams before the session limit", flush=True)
        for p in procs.values():
            if p.poll() is None:
                os.killpg(p.pid, signal.SIGTERM)
        time.sleep(30)
        break
progress()
# Done only when every runner exited 0: a stream stopped at the deadline can have logged episodes it never saved.
for seed, p in procs.items():
    print(f"stream {seed}: runner exit {p.poll()}")
ALL_DONE = all(p.poll() == 0 for p in procs.values())
# -15, 143 or still exiting (None): stopped here at the deadline (SIGTERM); any other non-zero code: the runner gave up after 3 crashes.
CRASHED = [seed for seed, p in procs.items() if p.poll() not in (0, -15, 143, None)]
if ALL_DONE:
    print("all streams finished")
elif CRASHED:
    print(f"streams {CRASHED} CRASHED (see seed_<s>/task_{TASK}/train.log): fix the cause before running a new version")
else:
    print("streams stopped at the deadline: add this version's output as input and Save & Run All again")

## 11. Analysis (when every stream is complete)

`analyze_noise_signal.py`: 3 Monte Carlo critics and 3 shuffled-noise controls on the 800 training episodes,
held-out action AUC with a bootstrap interval over the 200 held-out episodes, and the README's decision.

In [ ]:
if ALL_DONE:
    roots = " ".join(f"{OUT_ROOT}/seed_{s}/task_{TASK}" for s in STREAM_SEEDS)
    sh(f"cd {REPO} && {PY} -u noise_reachability/analyze_noise_signal.py --roots {roots} "
       f"--out {OUT_ROOT}/step3b_report.txt", env=RUN_ENV)
else:
    print("streams not finished; analysis skipped")

## 12. Files for the repo

In [ ]:
sh(f"cd '{OUT_ROOT}' && zip -q -r /kaggle/working/step3b_results.zip . -x '*.npz' '*.pt'")
if ALL_DONE:
    sh(f"cd '{OUT_ROOT}' && zip -q -r /kaggle/working/step3b_replay.zip . -i '*.npz' '*train_log.csv' '*provenance.jsonl'")
sh("du -h /kaggle/working/*.zip")